# Metagene analysis

This is the first step of the actual data analysis, and is used to determine appropriate offset for A or P site alignment. 

Requires Wig files from preprocessing & fasta/genbank for the genome used during alignment. Analysis only accounts for features labeled CDS in genbank.

**Begin with mapping_offset 0** to visualize raw wig files, **then choose appropriate offset value via 5'/3' metagene plots**. Change value, **restart kernel & run all cells again**. 
- Run in jupyter notebook with riboseq-env; should complete in <30 min & output plots + html log of the notebook + adjusted wig files (for nonzero mapping_offset) + cpm-normalized wig files

After choosing appropriate offset, move to **RiboSeq_GeneProfiling.ipynb** 

#### version info + references

2025-10
- RNAseq mapping offset always 0

2025-08
- adjusted to work with RNAseq for gene window plotting
- STILL NEED TO ADD TE WIG OUTPUT

2024-10-03
- added wig cpm conversion
  
2024-09-26

code references mainly [Mangano 2022](https://doi.org/10.1038/s41589-022-01138-9), plus [Mohammad 2019](https://doi.org/10.7554/eLife.42591) and [Zhang 2022](https://doi.org/10.1073/pnas.2118553119)


## Imports

In [ ]:
import os
import time
from collections import Counter
import numpy as np
import copy

import RiboSeq_Analysis_fxns as analysis
import RiboSeq_Analysis_plots as plot

from importlib import reload

starttime = time.time()

## User inputs


In [ ]:
user_inputs = { 
    
    # list names of all files to be analyzed - must have existing WIG files 
    # includes both RiboSeq & RNAseq
    'filenames': [ 'Rb1_DMSO', 'Rb1_TZD-10',  'Rb2_DMSO', 'Rb2_TZD-10', 
                   'Rn1_DMSO', 'Rn1_10xTZD',  'Rn2_DMSO', 'Rn2_10xTZD',  ],

    # currently only used to get a count of how many genes pass filters for all samples in a given replicate
    # but all metagene/gene plots after filter step can be easily modified to give replicate-specific figures using the genes_per_repetition dict
    'repNums': {'Rb1': ['Rb1_DMSO', 'Rb1_TZD-10', ], 
                'Rb2': ['Rb2_DMSO', 'Rb2_TZD-10', ],
                'Rn1': ['Rn1_DMSO', 'Rn1_10xTZD', ],
                'Rn2': ['Rn2_DMSO', 'Rn2_10xTZD', ], }, #{},

    'rnaSeq_pairs': {'Rb1_DMSO': 'Rn1_DMSO', 'Rb1_TZD-10': 'Rn1_10xTZD', 
                     
                      'Rb2_DMSO': 'Rn2_DMSO', 'Rb2_TZD-10': 'Rn2_10xTZD', 
                      
                    
                    'DMSO': 'Rn_DMSO', 'TZD-10': 'Rn_TZD-10', } , # { ribo1: rna1 },

    # must have a condition labeled either WT or DMSO to use as comparison for log fold change
    'condition_info': { 'DMSO': ['Rb1_DMSO', 'Rb2_DMSO'], 'TZD-10': ['Rb1_TZD-10','Rb2_TZD-10'], 
                        
                       'Rn_DMSO': ['Rn1_DMSO', 'Rn2_DMSO'], 'Rn_TZD-10': ['Rn1_10xTZD','Rn2_10xTZD'],
                        } , 
    
    # for each condition in condition info, label with either 'RiboSeq' or 'RNAseq'
    'condition_type': { 'DMSO': 'RiboSeq', 'TZD-10': 'RiboSeq', 
                        
                       'Rn_DMSO': 'RNAseq', 'Rn_TZD-10': 'RNAseq',
                        },
    
    # main directory for analysis & containing preprocessing files; analysis folder will be generated here
    'main_dir': '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/', 
    
    # directory containing your WIG files from preprocessing analysis for all samples in filenames
    'wig_dir':  '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/preprocessing/Wig/',

    # path of the genome fasta file - must be same fasta file as was used for genome alignment in preprocessing step
    'genome_fasta': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.fasta', 

    # path of the genbank file which matches the genome fasta file; features will be extracted from this
    'genome_genbank': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.gb', 
    
    # arbitrary value; used during visualization but not included in total sequence depth or avg gene reads so won't be included in normalization factor
    'utr_length_to_include': 50,
    
    # choose if you're using the Wig pileup at 5'/3' end of read or at center of read
    # In my experience 3pr gives much better peaks for both start & stop codon - also favored for bacterial riboSeq (Mohammad, Green, & Buskirk 2019)
    'alignment_type': '3pr', #'5pr' 'avg' or '3pr'
    
    # used for assigning P site; mapping offset =0 uses raw wig data
    # adjust the value after running the analysis so start codon peak aligns with 0 and stop codon peak aligns with -6 (stop codon in A-site at -3)
    # Note: mapping offset is generally a negative value -- JKDF05 riboSeq places P-site with -15 offset for 3pr pileup on both metagene plots
    'mapping_offset': -15,
    
    # used to plot normalized reads across a given window for a gene of interest -- ('geneName', [min, max])
    # note that if you include UTRs they must be ≤ the value for utr_length_to_include 
    # providing an empty list will give the full CDS window for the gene (no UTR) -- ('geneName', [])
    # 'genes_of_interest': [] 
    'genes_of_interest': [ ] ,

    # how long your shortest CDS can be
    'minCDS': 18,

    # number of nt at either end of genes to exclude from codon analysis
    'exclude_end_nt': 9
    
            }

In [ ]:
if len(user_inputs['rnaSeq_pairs'].keys()) != 0:
    RNAseq = True
else:
    RNAseq = False

## Running the metagene analysis

#### 1. Build analysis directories

In [ ]:
# check for & build the analysis directories

analysis_outdir = user_inputs['main_dir'] + time.strftime("%Y%m%d") + '_analysis/'
offset_dir = analysis_outdir + '{}_offset_{}/'.format(user_inputs['alignment_type'], user_inputs['mapping_offset'])
metagene_dir = offset_dir  + 'Metagene/'
fig_dir = metagene_dir + 'plots/'


if not os.path.exists(analysis_outdir):
    os.mkdir(analysis_outdir)
if not os.path.exists(offset_dir):
    os.mkdir(offset_dir)
if not os.path.exists(metagene_dir):
    os.mkdir(metagene_dir)
if not os.path.exists(fig_dir):
    os.mkdir(fig_dir)
    

#### 2. Load genome files

> parse_genome -->
>  - genome_dict 
>    - sequence dict contains full gene sequence with UTRs:
>      - full_sequence_dict = { featureName1 : [...ATG...TAG...], featureName2 : [...ATG...TAG...], ... } 
>    - strand dict contains info on + vs - strand for each gene:
>      - strand_dict = { featureName : 1, featureName2 : -1, ... }
>    - location dict contains genome positions of gene beginning & end:
>      -  location_dict = { featureName1 : (start, end),  ... }
>      -  not including UTRs
>    - total length of genome
>      -  total_genome_len = 4631469 (ex for CP009273.1)
>   
> **Includes only CDS features which are not listed as pseudogenes**
   

In [ ]:
# load in & check genome files


# Extract gene sequence & strand/location info from genome files FOR CDS & non-pseudogenes ONLY
# gene sequence includes UTRs of utr_length_to_include on either end
# gene location is the same as in genbank file (1st nt of start codon & 1 nt after stop codon)

genome_dict = analysis.parse_genome(user_inputs)

# sanity check that gene info dictionaries are the same size 
print('CDS feature count (excluding pseudogenes):', len(genome_dict['full_sequence_dict']))
print('gene info dictionaries are same size:', len(genome_dict['full_sequence_dict'].keys()) == len(genome_dict['strand_dict'].keys()) == len(genome_dict['location_dict'].keys()))

# check gene extraction
starts = []
stops = []
utr_length_to_include = user_inputs['utr_length_to_include']
for name,gene in list(genome_dict['full_sequence_dict'].items())[:]:
    starts.append(gene[utr_length_to_include:utr_length_to_include+3])
    stops.append(gene[-3-utr_length_to_include:-utr_length_to_include])

print('\nCount start & stop codons to check gene extraction')
print('####Start codons')
print(Counter(starts))
print('#####Stop codons')
print(Counter(stops))


#### 3. Load sample .wig files

>Read in forward & reverse wig files from preprocessing with alignment type selected above, shifting by the selected offset value. Mapping_offset=0 will give the raw wig files.

> read_in_wig_addOffset -->
> - for each sample, list of readcounts by position for each gene; 0 for positions without reads
>   - fwd positions + mapping_offset
>   - rev_positions - mapping offset
>   - accounts for reads which straddle the origin (circular genome)
> - feature_dict_meta = { sample_name1:  {gene1: [reads by position], gene2: [], ...}, sample_name2: {gene1: [reads by position], gene2: [], ... }
>   - includes readcounts for specified UTR length at both 5' and 3' ends

In [ ]:
# load in .wig files
reload(analysis)
mapping_offset = user_inputs['mapping_offset']
alignment_type = user_inputs['alignment_type']

sample_files = {}

for fname in user_inputs['filenames']:
    sample_files[fname] = (user_inputs['wig_dir'] + fname +'_%s_fwd_fromSam.wig' % alignment_type, 
                           user_inputs['wig_dir'] + fname +'_%s_rev_fromSam.wig' % alignment_type)

sample_names = sample_files.keys()

### feature_dict_meta = { sample_name: {gene1:[reads by position], gene2: [], ...}
feature_dict_meta = analysis.read_in_wig_addOffset(sample_files, genome_dict, user_inputs)


#### 4. Calculate sample sequencing depth for normalization

> Calculate sequencing depth (total readCounts across entire genome) for each sample
>   - Using **CDS portions only** (exclude UTRs for purpose of normalization to read depth)
>   - value should be somewhat smaller than total readcount from bowtie alignment due to this restriction
>
> *Any reads in overlapping gene regions will be double counted -- hopefully relatively minimal effect*

In [ ]:
# check total read numbers across CDS portions only -- this will be the value used for normalization
# should be somewhat smaller than total read counts from WIG conversion in your preprocessing logs

total_read_dict = {}
for sample in sample_names:
    all_features = []
    for read_list in feature_dict_meta[sample].values():
        all_features.extend(read_list[utr_length_to_include:-1*utr_length_to_include]) #read counts only in coding region - exclude UTR
    print(sample, np.sum(all_features))
    total_read_dict[sample] = np.sum(all_features)



#### 5. Periodicity check

> Ribosome profiling on *E.coli* with MNase digestion shows relatively minimal 3-nt periodicity
> ([Mohammad, Green, & Buskirk 2019](https://elifesciences.org/articles/42591#fig1)), 
> but still a good sanity check
>
> RNAseq should not show 3-nt periodicity

In [ ]:
#periodicity check
plot.plot_periodicity(sample_names, feature_dict_meta, fig_dir, user_inputs)

#### 6. Filter to remove low quality/expression genes from analysis

> produce a list of genes which pass coverage filters described in filter_genes function (for each individual sample)
> - updates feature_dict_meta[sample] to only those genes
> - note: filters selected based on Mangano 2022 and Zhang 2022
> - minCDS is modifiable but must be >100nt for 5' and 3' analysis

In [ ]:
# save an unfiltered copy of feature_dict_meta just in case
feature_dict_meta_full = copy.deepcopy(feature_dict_meta)

# remove genes which have low expression/coverage (>80% 0 or < 0.5 read average across CDS of gene) or have < 100nt CDS 
feature_dict_meta = analysis.filter_genes(sample_names, feature_dict_meta, user_inputs, minCDS=100)


#### 7. Create a list of genes common to all datasets

> list all genes (subject to the above constraints) which appear in all datasets <br>
> also performing this function for specified experimental repetition/condition sets, for comparison
>  - Not currently in use, but can supply rep & genes_per_repetition[rep]/ condition & genes_per_condition[condition] to plots past this point for specific analysis 

In [ ]:
genes_in_all_samples, genes_per_condition, genes_per_conditionPlusWT = analysis.common_genes(sample_names, feature_dict_meta, genome_dict, user_inputs, RNAseq)


#### 8. Metagene analysis - start & stop codon positioning

> With appropriate alignment, you should observe a clear peak for the start/stop codons due to ribosome accumulation during initiation/termination. 
> Use these plots to determine ideal offset.
> - fiveprime_meta: align the start codon peak with position 0 of the gene -- P-site alignment
> - threeprime_meta: align the stop codon peak with position 6 nt before end of gene -- P-site
>   - this is the P-site stall which puts the ribosome release factor in the A-site (-3 from end of gene)
> > Note: since we're working in python, positions are base 0 - meaning start codon is encoded at [0/1/2], 2nd aa at [3/4/5]... <br>
> > also, list indexing is noninclusive, so ['a', 'b', 'c', 'd'][0:3] gives a list [a,b,c] <br>
> > THUS stop codon is at [-3/-2/-1] so that [start:stop] encodes full length of the protein
> > and individual amino acids (if using 1-based positioning, which is standard) are located at [(aa_position -1)*3:(aa_position -1)*3+3]
>
> P-value calculated using [mixed linear model](https://www.statsmodels.org/stable/generated/statsmodels.regression.mixed_linear_model.MixedLM.html#statsmodels.regression.mixed_linear_model.MixedLM) -- [explanation](https://stats.oarc.ucla.edu/other/mult-pkg/introduction-to-linear-mixed-models/)
> - in this case: accounting for differences between samples & by gene
>   
> note: winsorize - from Mangano 2022 code
> "transforms the data slightly to mitigate the effect of outliers within each gene before subsequently taking column-averages across the genes for each position. It's not clear whether this is necessary (or helpful) but I found it to be a useful robustness check to ensure that the results are not dependent on findings from a few strongly outlying data points/peaks. Findings seem very robust regardless of the flag."

In [ ]:
# determine appropriate mapping offset to place the start codon peak at 0 (--> P-site alignment) and stop codon peak at -6 
# stop codon peak at -6 is the P-site stall which puts the ribosome release factor in the A-site (-3 from end of gene)
# normalized to mean reads for the gene and total sequencing depth
# note: this runs a little slower due to p-value calculation
reload(plot)
if len(user_inputs['rnaSeq_pairs']) != 0:
    rs = [s for s in sample_names if s in user_inputs['rnaSeq_pairs'].keys()]
    plot.plot_fiveprime_meta(feature_dict_meta, rs, genes_in_all_samples, fig_dir, user_inputs, total_read_dict)
    plot.plot_threeprime_meta(feature_dict_meta, rs, genes_in_all_samples, fig_dir, user_inputs, total_read_dict)

else: 
    plot.plot_fiveprime_meta(feature_dict_meta, sample_names, genes_in_all_samples, fig_dir, user_inputs, total_read_dict)
    plot.plot_threeprime_meta(feature_dict_meta, sample_names, genes_in_all_samples, fig_dir, user_inputs, total_read_dict)


#### 8b. Lengthwise Metagene analysis - reallocation of reads in gene window
> plotting positions as a fraction of gene length (rounded to 2 decimal places);
> reads at each fractional position are averaged across ORFs

In [ ]:
if len(user_inputs['rnaSeq_pairs']) >0:
    plot.lengthwise_metagene(sample_names, genes_in_all_samples, feature_dict_meta, fig_dir, user_inputs, 'RiboSeq')
else:
    plot.lengthwise_metagene(sample_names, genes_in_all_samples, feature_dict_meta, fig_dir, user_inputs)

#### 9. Verify offset via individual genes of interest

> This chunk will plot genes & windows specified in user_inputs with the format ( geneName, [window start, window end] ) or full gene CDS for ( geneName, [] )
> <br> By looking specifically at genes which commonly stall under DMSO/WT conditions, this can be used to verify the selected mapping offset
> <br> Can also be used just to look at genes of interest once stall sites have been identified
>
> secM has a known stalling sequence which should pause ribosomes with G165 located in the P-site and P166 in the A-site --> Check whether we observe this peak for this single gene: ( secM, [ 480 , 513 ] ) <br> We would of course expect to see it in the ribosome profiling reads and not the RNA-seq reads. <br>
> aaPosition (1-based) --> nt position (0-based, 3nt/aa) 165 --> [(165-1)*3 : (165-1)*3 + 3] = [492, 493, 494] G165 --> [GGC]
>
> Note: may be better to assign reads to 2nd nt of the codon so that slight deviations (+/- 1) still assign reads to the same P-site codon, but makes peaks appear shifted by 1nt


In [ ]:
# check offset
reload(plot)
for gene_window in user_inputs['genes_of_interest']:
    gene=gene_window[0]
    window=gene_window[1]
    
    # get the full name of a gene of interest
    geneID = analysis.gene_lookup(gene, genome_dict)

    #supply full gene name to plot reads for that gene
    if len(window) == 2:
        plot.short_CDS_window_rpkm(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, fig_dir, user_inputs, window)
        plot.short_CDS_window_TE(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, fig_dir, user_inputs, window, RNAseq)

    else: 
        plot.whole_gene_window_rpkm(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, fig_dir, user_inputs)
        plot.whole_gene_window_TE(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, fig_dir, user_inputs, RNAseq)
        

#### 9b. Verify offset via individual genes of interest

> Producing stacked plots of full gene windows of genes of interest; TE values averaged across condition 

In [ ]:
reload(plot)
if RNAseq == True:
    for gene in set([gene for gene, pos in user_inputs['genes_of_interest']]):
        plot.stack_plot_TE(gene,[condition for condition in user_inputs['condition_info'] if user_inputs['condition_type'][condition] == 'RiboSeq'],
                           fig_dir, feature_dict_meta, total_read_dict, genome_dict, user_inputs)

else:
    for gene in set([gene for gene, pos in user_inputs['genes_of_interest']]):
        plot.stack_plot(gene, [condition for condition in user_inputs['condition_info'] if user_inputs['condition_type'][condition] == 'RiboSeq'],  
                        fig_dir, feature_dict_meta, total_read_dict, genome_dict, user_inputs)

#### 10. Rewrite wig files to incorporate mapping offset

> modified wig files will be saved in a subfolder of the offset_dir -- original Wig files are left unchanged.
> <br>If mapping offset is 0 no new Wig files will be produced
> - new wig files are not currently used for analysis, but can be compared with the raw wig file to see if offset is properly applied & are useful for IGV viewing

In [ ]:

# if mapping offset ≠ 0: rewrite wig files to include mapping offset & calculate cpm for the modified wig files
# if mapping offset = 0: do not rewrite wig files; only calculate cpm for the raw wig files

if mapping_offset !=0:

    # make a folder to output modified wig
    adjusted_wig_outdir = metagene_dir + 'Wig/'
    if not os.path.exists(adjusted_wig_outdir):
        os.mkdir(adjusted_wig_outdir)
        
    # write wig file with offset 
    analysis.rewrite_wig(sample_files, user_inputs, genome_dict, adjusted_wig_outdir)
    
    # rewrite sample file dict to call newly modified wig files
    sample_files = {}
    for fname in user_inputs['filenames']:
        sample_files[fname] = (adjusted_wig_outdir + fname + '_%s_fwd_offset_%s.wig' % (alignment_type, mapping_offset), 
                               adjusted_wig_outdir + fname + '_%s_rev_offset_%s.wig' % (alignment_type, mapping_offset))
    
# for any mapping offset: create cpmWig outdir and rewrite wig files with counts --> cpm
cpm_outdir = metagene_dir + 'cpm_Wig/' 
if not os.path.exists(cpm_outdir):
    os.mkdir(cpm_outdir)

analysis.wig_to_cpm(sample_files, total_read_dict, user_inputs, cpm_outdir)

#may want to add conversion to TE at some point

#### 11. Close out & save analysis

> **From here, move to RiboSeq_GeneProfiling.ipynb**

In [ ]:
endtime = time.time()
print('Metagene analysis complete on {} samples in {} hh:mm:ss - alignment type: {}, mapping_offset: {}'.format(len(sample_names), time.strftime("%H:%M:%S", time.gmtime(endtime-starttime)), user_inputs['alignment_type'], user_inputs['mapping_offset']))


In [ ]:
# sleep to allow notebook time to autosave all outputs 
time.sleep(120)
# produce an html copy of the notebook ; this is your log of the run & contains all code + cell outputs + plots 
os.system("jupyter nbconvert --to html RiboSeq_MetageneAnalysis.ipynb")
os.rename('RiboSeq_MetageneAnalysis.html', 
          metagene_dir + 'RiboSeq_MetageneAnalysis_notebookLog_{}Alignment_{}offset.html'.format(user_inputs['alignment_type'], user_inputs['mapping_offset']))

# **Next: RiboSeq_GeneProfiling.ipynb**